In [ ]:
#乳がんの診断データセットを用いて細胞のデータから悪性か良性かを見分ける
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np
import japanize_matplotlib

data = load_breast_cancer()

df = pd.DataFrame(data.data,columns=data.feature_names)
df['target'] = data.target
y=df['target'] #腫瘍が悪性(がん)０か良性１か
x=df.drop(columns=['target'])
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

#剪定なし(過学習)
clf_none = DecisionTreeClassifier(random_state=42)
clf_none.fit(x_train,y_train)

#正解率accuracyを表示。1=100%正解(過学習)という意味
#決定木において訓練100%正解は過学習のサイン：これは「データ全体の傾向（本質）」を学んだのではなく、「426人分のデータを1人残らず完全に仕分けるためのマニアックなif文」を100%丸暗記した状態
print(f"訓練スコア(制限なし)：{clf_none.score(x_train,y_train):.3f}")
print(f"テストスコア(制限なし)：{clf_none.score(x_test,y_test):.3f}")

#剪定なし結果を視覚化
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20,10))
plot_tree(clf_none,feature_names=data.feature_names,class_names=data.target_names,filled=True)
plt.show()

In [ ]:
import japanize_matplotlib
import matplotlib.pyplot as plt

# 1から15までの深さでスコアの動きを追跡する
depths = range(1, 16)
train_scores = []
test_scores = []

for depth in depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(x_train, y_train)
    train_scores.append(clf.score(x_train, y_train))
    test_scores.append(clf.score(x_test, y_test))

# グラフのプロット
plt.figure(figsize=(10, 6))
plt.plot(
    depths,
    train_scores,
    label="訓練スコア (Train Accuracy)",
    marker="o",
    linestyle="--",
    color="blue",
)
plt.plot(
    depths,
    test_scores,
    label="テストスコア (Test Accuracy)",
    marker="o",
    linewidth=2,
    color="orange",
)

# 剪定なし（過学習状態）の位置に補助線を引いて目立たせる
plt.axvline(x=clf_none.get_depth(), color="red", linestyle=":", label="剪定なしの深さ")

plt.xlabel("max_depth (木の深さ)", fontsize=12)
plt.ylabel("Accuracy (正解率)", fontsize=12)
plt.title("決定木の深さと過学習の関係（検証曲線）", fontsize=14)
plt.xticks(depths)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"剪定なしモデルの実際の木の深さ: {clf_none.get_depth()}")

In [ ]:
#交差検証の事後剪定
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

# 1. まず、制限なしの木を一度作り、このデータから発生し得る「ccp_alphaの全候補」を逆算させる
clf_base = DecisionTreeClassifier(random_state=42)
path = clf_base.cost_complexity_pruning_path(x_train, y_train)
ccp_alphas_candidates = path.ccp_alphas

print(f"AIが逆算したアルファの候補数: {len(ccp_alphas_candidates)} 個")
# ※マイナスの値や異常値を取り除くため、念のため0以上の値に限定する
ccp_alphas_candidates = [a for a in ccp_alphas_candidates if a >= 0]

# 2. 逆算した候補リストを、そのままグリッドサーチの探索範囲にセットする
param_grid = {
    'ccp_alpha': ccp_alphas_candidates
}

# 3. 3分割交差検証（cv=3）を設定して、この候補の中から「平均スコアが最高のアルファ」を全探索
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42), 
    param_grid, 
    cv=3, 
    scoring='accuracy'
)

# 4. 交差検証を実行
grid_search.fit(x, y)

print("\n--- 探索完了 ---")
print(f"交差検証で選ばれた、最終のハイパーパラメータ: {grid_search.best_params_['ccp_alpha']:.6f}")
print(f"そのときの事後剪定の最高平均スコア: {grid_search.best_score_:.4f}")

# （先ほどのgrid_search.fit(x, y) の後ろにこれを追加）

# 1. 交差検証で選ばれた「最高の一本（モデル）」をピンポイントで抜き出す
best_clf_post = grid_search.best_estimator_

# 2. 抜き出した最強の木をプロットする
plt.figure(figsize=(20,10))
plot_tree(best_clf_post, feature_names=data.feature_names, class_names=data.target_names, filled=True)
plt.title(f"事後剪定ツリー (ccp_alpha = {grid_search.best_params_['ccp_alpha']:.6f})", fontsize=16)
plt.show()

In [ ]:
import japanize_matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# 1. 探索する深さの候補（1から10）
depths = range(1, 11)
param_grid_pre = {"max_depth": depths}

# 2. グリッドサーチを設定
# (return_train_score=True にすることで、グラフに必要な訓練スコアも一緒に記録してくれます)
grid_search_pre = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_pre,
    cv=3,
    scoring="accuracy",
    return_train_score=True,
)

# 3. 交差検証を実行
grid_search_pre.fit(x, y)

# 4. グリッドサーチの内部に保存されている「全実験のスコア詳細」を表（DataFrame）として取り出す
cv_results = pd.DataFrame(grid_search_pre.cv_results_)

# 3回分の交差検証の「平均スコア」をそれぞれ抽出
mean_train_scores = cv_results["mean_train_score"]
mean_test_scores = cv_results["mean_test_score"]

# 5. 結果を折れ線グラフとしてプロット
plt.figure(figsize=(10, 6))
plt.plot(
    depths,
    mean_train_scores,
    label="Train Accuracy (交差検証・訓練平均)",
    marker="o",
    linestyle="--",
)
plt.plot(
    depths,
    mean_test_scores,
    label="Test Accuracy (交差検証・テスト平均)",
    marker="o",
    linewidth=2,
)

plt.xlabel("max_depth (木の深さ)", fontsize=12)
plt.ylabel("Accuracy (正解率)", fontsize=12)
plt.title(
    "Validation Curve for Decision Tree (3-Fold Cross Validation)", fontsize=14
)
plt.xticks(depths)
plt.legend()
plt.grid(True, alpha=0.3)  # さっき覚えた薄い目盛り線！
plt.show()

# 画面にもテキストで最高スコアを出力
print("--- 探索完了 ---")
print(
    f"交差検証で選ばれた、真に最適な深さ(max_depth): {grid_search_pre.best_params_['max_depth']}"
)
print(f"そのときの最高平均テストスコア: {grid_search_pre.best_score_:.4f}")

In [ ]:
# 交差検証事前剪定
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree

# ① 事前剪定モデル（深さ3固定）の条件下で、グリッドサーチを設定
# （cv=3で3分割交差検証を行います）
param_grid_pre = {"max_depth": [5]}

grid_search_pre = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_pre,
    cv=3,
    scoring="accuracy",
)

# ② 交差検証を実行し、裏でデータ全体を使ってモデルを自動で育て直す
grid_search_pre.fit(x, y)

# ③ 3回の交差検証の「平均スコア」を表示（さっき画面に出ていた0.9156が出ます）
print(f"事前剪定（交差検証平均スコア）: {grid_search_pre.best_score_:.4f}")

# ④ 交差検証を経て完成した「最高の一本」をピンポイントで抜き出す
best_clf_pre = grid_search_pre.best_estimator_

# ⑤ 抜き出した「真の事前剪定ツリー」をプロット
plt.figure(figsize=(20, 10))
plot_tree(
    best_clf_pre,
    feature_names=data.feature_names,
    class_names=data.target_names,
    filled=True,
)
plt.title("事前剪定ツリー (max_depth = 3)", fontsize=16)
plt.show()

In [ ]:
# 深さの候補（1から10まで）
#testのグラフが一番高いところを木の深さとして設定する
import japanize_matplotlib

depths = range(1, 11)
train_scores = []
test_scores = []

for depth in depths:
    # それぞれの深さでモデルを作る
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(x_train, y_train)
    
    # スコアを記録
    train_scores.append(clf.score(x_train, y_train))
    test_scores.append(clf.score(x_test, y_test))

# 結果をグラフにする
plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, label='Train Accuracy', marker='o', linestyle='--')
plt.plot(depths, test_scores, label='Test Accuracy', marker='o', linewidth=2)
plt.xlabel('max_depth (木の深さ)', fontsize=12)
plt.ylabel('Accuracy (正解率)', fontsize=12)
plt.title('Validation Curve for Decision Tree', fontsize=14)
plt.xticks(depths)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#木の深さを３に制限して、事前剪定を行う
clf_pruned = DecisionTreeClassifier(max_depth=3,random_state=42)

#訓練データで学習
clf_pruned.fit(x_train,y_train)

#剪定後の結果
print("---事前剪定(max_depth=3)の結果")
print(f"訓練スコア：{clf_pruned.score(x_train,y_train):.3f}")
print(f"テストスコア：{clf_pruned.score(x_test,y_test):.3f}")
plt.figure(figsize=(25,13))
plot_tree(clf_pruned,feature_names=data.feature_names,class_names=data.target_names,filled=True,rounded=True,fontsize=12)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from matplotlib.colors import ListedColormap

# --- 1. 最適な事後剪定のパラメータ（alpha）を見つける ---
# 一度、制限なしでモデルを作ります
clf_full = DecisionTreeClassifier(random_state=42)
clf_full.fit(x_train, y_train)

# パラメータ「ccp_alpha」の候補リストを自動計算で取得します
path = clf_full.cost_complexity_pruning_path(x_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# 各alphaの値で実際にモデルを作って、テストスコアが一番高くなるalphaを探します
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=ccp_alpha)
    clf.fit(x_train, y_train)
    clfs.append(clf)

# テストデータに対する正解率が最大になるモデルを抽出
test_scores = [clf.score(x_test, y_test) for clf in clfs]
best_clf_index = np.argmax(test_scores)
clf_post = clfs[best_clf_index]
best_alpha = ccp_alphas[best_clf_index]

print(f"選ばれた最高のハイパーパラメータ: {best_alpha:.4f}")
print(f"事後剪定後の訓練スコア: {clf_post.score(x_train, y_train):.3f}")
print(f"事後剪定後のテストスコア: {clf_post.score(x_test, y_test):.3f}")

plt.figure(figsize=(20,13))
plot_tree(clf_post,feature_names=data.feature_names,class_names=data.target_names,filled=True)
plt.show()


In [ ]:
print(len(df))